# CNN with PyTorch — AI Engineering Interview Prep

Convolutional Neural Network on MNIST using PyTorch.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## 1. Load MNIST Dataset

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean/std
])

train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_data,  batch_size=1000, shuffle=False, num_workers=0)

print(f"Train: {len(train_data)} | Test: {len(test_data)}")

# Visualise some images
examples = iter(train_loader)
imgs, labels = next(examples)

fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for i, ax in enumerate(axes):
    ax.imshow(imgs[i].squeeze(), cmap='gray')
    ax.set_title(labels[i].item())
    ax.axis('off')
plt.suptitle("MNIST Samples")
plt.show()

## 2. Define CNN Architecture

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Conv block 1: 1→32 channels, 3x3 kernel
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)
        # Conv block 2: 32→64 channels
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)
        # After 2 maxpool(2,2): 28x28 → 7x7
        self.fc1   = nn.Linear(64 * 7 * 7, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        # Block 1
        x = F.relu(self.bn1(self.conv1(x)))   # 28x28x32
        x = F.max_pool2d(x, 2)                # 14x14x32
        # Block 2
        x = F.relu(self.bn2(self.conv2(x)))   # 14x14x64
        x = F.max_pool2d(x, 2)                # 7x7x64
        # Classifier
        x = x.flatten(1)                      # 7*7*64 = 3136
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)  # raw logits

model = CNN().to(device)
print(model)

# Count parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {n_params:,}")

## 3. Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(imgs)
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            total_loss += criterion(outputs, labels).item() * len(imgs)
            correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

n_epochs = 5
for epoch in range(1, n_epochs + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    va_loss, va_acc = evaluate(model, test_loader, criterion)
    scheduler.step()
    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss)
    history['val_acc'].append(va_acc)
    print(f"Epoch {epoch}/{n_epochs}: train_acc={tr_acc:.4f}, val_acc={va_acc:.4f}")

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Val')
ax2.set_title('Accuracy'); ax2.legend()
plt.tight_layout()
plt.show()

print(f"Final test accuracy: {history['val_acc'][-1]:.4f}")

## 4. Convolution Intuition

In [ ]:
# Manual convolution — show what edge detector filter does
edge_filter = torch.tensor([[-1., -1., -1.],
                              [-1.,  8., -1.],
                              [-1., -1., -1.]]).view(1, 1, 3, 3)

# Take one image from test set
sample_img, sample_label = next(iter(test_loader))
img = sample_img[:1]  # shape (1, 1, 28, 28)

with torch.no_grad():
    filtered = F.conv2d(img, edge_filter, padding=1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(img.squeeze().numpy(), cmap='gray'); ax1.set_title(f"Original ({sample_label[0].item()})"); ax1.axis('off')
ax2.imshow(filtered.squeeze().numpy(), cmap='gray'); ax2.set_title('After Edge Filter'); ax2.axis('off')
plt.show()

## Interview Tips

- **Conv2d params**: `in_channels, out_channels, kernel_size, stride=1, padding=0`.
- **Output size**: `floor((W + 2P - K)/S) + 1`. With padding=1, kernel=3, stride=1: size preserved.
- **Receptive field**: grows with depth. Layer 1 sees 3x3; Layer 2 sees 5x5; etc.
- **BatchNorm**: normalizes activations per mini-batch. Stabilizes training, reduces sensitivity to lr.
- **Dropout**: randomly zeros activations during training — reduces co-adaptation between neurons.
- **Transfer learning**: use pretrained (ImageNet) CNN backbone; fine-tune last layers on your data.